# OneVoice V2 — MT benchmark
Đo baseline cả hai chiều và candidate EnViT5 VI→EN trên test/minimal/safety. Candidate chỉ là ứng viên cho đến khi qua held-out quality gate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'model_cache/huggingface')
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
# EnViT5's SentencePiece tokenizer is incompatible with Transformers 5.x. Force the tested 4.x stack even if another notebook upgraded it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.2.6', 'PyYAML', 'torch', 'sacremoses'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0'], check=True)
version_check = subprocess.run([sys.executable, '-c', "import transformers, tokenizers, sentencepiece; print('MT versions:', transformers.__version__, tokenizers.__version__, sentencepiece.__version__); assert transformers.__version__ == '4.57.1'"], check=True, text=True, capture_output=True)
print(version_check.stdout, end='')
# MT is text-only. Remove Colab's optional torchvision when its binary is incompatible with torch; Transformers otherwise imports it while loading T5.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
REPORT_ROOT = DRIVE_ROOT / 'reports/mt'
# Prefer the durable Drive checkpoint. The Hugging Face copy may be private and unavailable from another Colab account.
CANDIDATE_LOCAL_DIRS = (
    DRIVE_ROOT / 'models/envit5_finetuned_vi2en_v2/best',
    DRIVE_ROOT / 'models/envit5_finetuned_construction/best',
)
CANDIDATE_VI2EN = next((str(path) for path in CANDIDATE_LOCAL_DIRS if (path / 'config.json').is_file()), None)
if CANDIDATE_VI2EN:
    print('Using persistent VI→EN candidate:', CANDIDATE_VI2EN)
else:
    print('No local VI→EN candidate found; baseline runs remain available. Restore a Drive checkpoint before candidate evaluation.')

def run_streaming(command, label):
    print(f'\n[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    print(f'[{label}] exit code: {code}', flush=True)
    if code:
        raise RuntimeError(f'{label} failed; the complete subprocess log is printed above.')

print('Source:', REPO, '| Reports:', REPORT_ROOT)


In [ ]:
# A Colab runtime or another notebook can reinstall an incompatible torchvision after setup. Remove it immediately before any T5 subprocess starts.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
EN2VI_CANDIDATE_DIR = DRIVE_ROOT / 'models/envit5_finetuned_en2vi_v1/best'
CANDIDATE_EN2VI = str(EN2VI_CANDIDATE_DIR) if (EN2VI_CANDIDATE_DIR / 'config.json').is_file() else None
if CANDIDATE_EN2VI:
    print('Using persistent EN→VI candidate:', CANDIDATE_EN2VI)
else:
    print('No local EN→VI candidate yet; EN→VI baseline only.')
MODEL_RUNS = [('base', ('vi2en', 'en2vi'), None)]
if CANDIDATE_VI2EN:
    MODEL_RUNS.append(('candidate_vi2en', ('vi2en',), CANDIDATE_VI2EN))
if CANDIDATE_EN2VI:
    MODEL_RUNS.append(('candidate_en2vi', ('en2vi',), CANDIDATE_EN2VI))
for model_label, directions, model_source in MODEL_RUNS:
    for direction in directions:
        for suite in ('test', 'minimal', 'safety'):
            for mode in ('raw', 'context'):
                report_dir = REPORT_ROOT / model_label / direction / suite / mode
                label = f'MT {model_label}/{direction}/{suite}/{mode}'
                required = ('aggregate.json', 'predictions.csv', 'run_manifest.json')
                if all((report_dir / name).is_file() for name in required):
                    print(f'[{label}] already complete on Drive; skipping.', flush=True)
                    continue
                command = [sys.executable, 'scripts/benchmark_mt_v2.py', '--direction', direction, '--suite', suite, '--progress-every', '25', '--report-dir', str(report_dir)]
                if model_source:
                    command.extend(['--model-source', model_source])
                if mode == 'context':
                    command.append('--with-context')
                run_streaming(command, label)


In [ ]:
import json
{f'{model_label}/{direction}/{suite}/{mode}': json.loads((REPORT_ROOT / model_label / direction / suite / mode / 'aggregate.json').read_text(encoding='utf-8')) for model_label, directions, _ in MODEL_RUNS for direction in directions for suite in ('test', 'minimal', 'safety') for mode in ('raw', 'context')}


In [ ]:
DASHBOARD = DRIVE_ROOT / 'reports/benchmark_dashboard.md'
subprocess.run([sys.executable, 'scripts/build_benchmark_dashboard.py', '--report-root', str(DRIVE_ROOT / 'reports'), '--output', str(DASHBOARD)], check=True)
print(DASHBOARD.read_text(encoding='utf-8'))
